# 13 · Leave-One-Dataset-Out — OAI Held Out (3 seeds)

Trains the full method on NHANES III, MRKR, and Mendeley, and evaluates on the held-out OAI set across three random seeds.

## Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
import sys, importlib
sys.path.insert(0,'/content/drive/MyDrive/Master Thesis/scope3')
import config; importlib.reload(config)
import pandas as pd, numpy as np
from pathlib import Path
import torch
if 'training_lib' in sys.modules: importlib.reload(sys.modules['training_lib'])
import training_lib as T
if not torch.cuda.is_available(): raise RuntimeError('No GPU. Runtime -> Change runtime type -> GPU.')
print('GPU:', torch.cuda.get_device_name(0))
manifest = T.prepare_local_data()
print(f'Manifest: {len(manifest):,} rows')

Mounted at /content/drive
GPU: NVIDIA A100-SXM4-40GB
Copied array in 33s
Loaded array (61558, 224, 224) in 1s
Manifest: 61,558 rows


In [ ]:
def make_splits(manifest, test_dataset, seed=0):
    pool=manifest[manifest['dataset']!=test_dataset].copy()
    test=manifest[manifest['dataset']==test_dataset].copy()
    if 'split' in pool.columns and pool['split'].isin(['train','val']).any():
        tr=pool[pool['split'].isin(['train','TRAIN'])]; va=pool[pool['split'].isin(['val','VAL','validation'])]
        if len(va)==0: va=pool.sample(frac=0.15,random_state=seed); tr=pool.drop(va.index)
    else:
        va=pool.sample(frac=0.15,random_state=seed); tr=pool.drop(va.index)
    return tr.reset_index(drop=True), va.reset_index(drop=True), test.reset_index(drop=True)
print('Split helper ready.')

Split helper ready.


## Execution

In [ ]:
fk='fold4_test_oai'; test_ds=config.LODO_FOLDS[fk]; mech=dict(config.FULL_METHOD)
try:
    _sw=pd.read_csv(str(config.RESULTS_DIR/'grl_lambda_sweep.csv'))
    _best=float(_sw.loc[_sw['external_acc5'].idxmax(),'grl_lambda_max'])
    mech['grl_lambda_max']=_best; print(f'Using swept GRL lambda: {_best}')
except Exception: print('No sweep file; using config GRL_LAMBDA_MAX')
results=[]
for seed in config.SEEDS:
    run=f'{fk}_seed{seed}'
    tr,va,te=make_splits(manifest,test_ds,seed=seed)
    print(f'--- {run} (train={len(tr):,} val={len(va):,} test={len(te):,}) ---',flush=True)
    r=T.run_training(run,tr,va,te,mech,seed,config.CKPT_DIR,config.RESULTS_DIR,log_fn=print)
    results.append(r)
df=pd.DataFrame(results)
print(f'External acc5: {df["external_acc5"].mean():.4f} +/- {df["external_acc5"].std():.4f}')
print(f'External QWK : {df["external_qwk"].mean():.4f}')
print(f'Gap          : {df["gap"].mean():.4f}')
df.to_csv(str(config.RESULTS_DIR/f'{fk}_results.csv'),index=False)

Using swept GRL lambda: 0.5
--- fold4_test_oai_seed0 (train=45,059 val=7,952 test=8,547) ---
Downloading: "https://download.pytorch.org/models/convnext_large-ea097f82.pth" to /root/.cache/torch/hub/checkpoints/convnext_large-ea097f82.pth


100%|██████████| 755M/755M [00:03<00:00, 222MB/s]


  [fold4_test_oai_seed0] ep1/18 loss=2.988 tr=0.477 val=0.424 gap=+0.053 qwk=0.243 grl=0.00 (122s)
  [fold4_test_oai_seed0] ep2/18 loss=2.507 tr=0.529 val=0.447 gap=+0.082 qwk=0.357 grl=0.14 (105s)
  [fold4_test_oai_seed0] ep3/18 loss=2.324 tr=0.550 val=0.473 gap=+0.077 qwk=0.471 grl=0.25 (106s)
  [fold4_test_oai_seed0] ep4/18 loss=2.209 tr=0.570 val=0.487 gap=+0.083 qwk=0.508 grl=0.34 (105s)
  [fold4_test_oai_seed0] ep5/18 loss=2.134 tr=0.584 val=0.483 gap=+0.101 qwk=0.500 grl=0.40 (106s)
  [fold4_test_oai_seed0] ep6/18 loss=2.040 tr=0.586 val=0.492 gap=+0.094 qwk=0.489 grl=0.44 (105s)
  [fold4_test_oai_seed0] ep7/18 loss=1.974 tr=0.586 val=0.503 gap=+0.083 qwk=0.515 grl=0.47 (106s)
  [fold4_test_oai_seed0] ep8/18 loss=1.888 tr=0.587 val=0.510 gap=+0.077 qwk=0.537 grl=0.48 (106s)
  [fold4_test_oai_seed0] ep9/18 loss=1.819 tr=0.594 val=0.502 gap=+0.092 qwk=0.531 grl=0.49 (106s)
  [fold4_test_oai_seed0] ep10/18 loss=1.774 tr=0.584 val=0.519 gap=+0.065 qwk=0.556 grl=0.49 (105s)
  [fold4_